In [ ]:
#Qué hace: define el outcome principal: respuesta clínica sostenida (RCS) de forma operativa y longitudinal.
#Clave: fija qué es “mejoría” y qué significa “sostenida”.

In [13]:
# ============================================================
# 10_respuesta_clinica_sostenida.ipynb
#
# What changes vs previous:
#   - START_DAY_IDX = 1 (exclude day 0 baseline)
#   - clinical_ok_observed requires BOTH physiology AND labs (AND)
#   - MIN_CONSECUTIVE_DAYS configurable (default=2)
#
# Input:
#   - 04_cohorte_base_T0.parquet
#   - 07_dominios_fisiologia.parquet
#   - 08_dominios_labs.parquet (or alternative names)
# Output:
#   - 10_respuesta_clinica_sostenida.parquet
# ============================================================

import os
import pandas as pd
import numpy as np

In [14]:
# -----------------------------
# 0) Paths
# -----------------------------
COHORT_PATH = "04_cohorte_base_T0.parquet"
PHYS_PATH   = "07_dominios_clinicos.parquet"

LABS_CANDIDATES = ["08_labs_diarios.parquet"]

OUT_PATH = "10_respuesta_clinica_sostenida.parquet"

In [15]:
# -----------------------------
# 1) Clinical definition knobs
# -----------------------------
START_DAY_IDX = 1          # <-- IMPORTANT: exclude day 0 (baseline)
MIN_CONSECUTIVE_DAYS = 3   # 3 consecutive days with improvement
REQUIRE_BOTH_DOMAINS = True  # True = phys AND labs ; False = phys OR labs

In [16]:
# -----------------------------
# 2) Helpers
# -----------------------------
def require_cols(df, cols, name="df"):
    missing = set(cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {name}: {missing}")

def to_int64(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=cols).copy()
    for c in cols:
        df[c] = df[c].astype("int64")
    return df

def pick_existing_file(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

def first_valid_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def build_observed_ok(df, ok_cols):
    """
    ok_cols: binary columns with possible NaN
    returns:
      - ok_observed: 1 if >=1 measured and >=1 ok else NaN
    """
    df = df.copy()
    df["n_measured"] = df[ok_cols].notna().sum(axis=1)
    df["n_ok"] = df[ok_cols].sum(axis=1, skipna=True)
    df["ok_observed"] = np.where(
        (df["n_measured"] >= 1) & (df["n_ok"] >= 1),
        1,
        np.nan
    )
    return df

def sustained_run_first_day(x, min_consecutive=2):
    run = 0
    for i, v in enumerate(x):
        if v == 1:
            run += 1
            if run >= min_consecutive:
                return i - (min_consecutive - 1)
        else:
            run = 0
    return np.nan

In [17]:
# -----------------------------
# 3) Load
# -----------------------------
if not os.path.exists(COHORT_PATH):
    raise FileNotFoundError(f"Missing file: {COHORT_PATH}")
if not os.path.exists(PHYS_PATH):
    raise FileNotFoundError(f"Missing file: {PHYS_PATH}")

labs_path = pick_existing_file(LABS_CANDIDATES)
if labs_path is None:
    raise FileNotFoundError(
        "Could not find labs file. Tried:\n  - " + "\n  - ".join(LABS_CANDIDATES)
    )

df_cohort = pd.read_parquet(COHORT_PATH)
df_phys   = pd.read_parquet(PHYS_PATH)
df_labs   = pd.read_parquet(labs_path)

print("Loaded:")
print("  cohort:", df_cohort.shape, COHORT_PATH)
print("  phys  :", df_phys.shape, PHYS_PATH)
print("  labs  :", df_labs.shape, labs_path)

Loaded:
  cohort: (82668, 12) 04_cohorte_base_T0.parquet
  phys  : (126791, 13) 07_dominios_clinicos.parquet
  labs  : (170806, 12) 08_labs_diarios.parquet


In [18]:
# -----------------------------
# 4) Standardize keys
# -----------------------------
KEYS = ["subject_id", "hadm_id", "icu_stay_id", "day_idx"]
require_cols(df_phys, KEYS, "df_phys")
require_cols(df_labs, KEYS, "df_labs")

df_phys = to_int64(df_phys, ["subject_id", "hadm_id", "icu_stay_id"])
df_labs = to_int64(df_labs, ["subject_id", "hadm_id", "icu_stay_id"])

df_phys["day_idx"] = pd.to_numeric(df_phys["day_idx"], errors="coerce").astype("Int64")
df_labs["day_idx"] = pd.to_numeric(df_labs["day_idx"], errors="coerce").astype("Int64")

df_phys = df_phys.dropna(subset=["day_idx"]).copy()
df_labs = df_labs.dropna(subset=["day_idx"]).copy()

df_phys["day_idx"] = df_phys["day_idx"].astype("int64")
df_labs["day_idx"] = df_labs["day_idx"].astype("int64")

In [19]:
# -----------------------------
# 5) Detect domain columns
# -----------------------------
phys_agg_col = first_valid_col(df_phys, ["physio_ok_observed", "physio_ok_all", "physio_ok_daily", "physio_ok"])
phys_component_cols = [c for c in ["temp_ok", "map_ok", "sf_ok"] if c in df_phys.columns]

labs_agg_col = first_valid_col(df_labs, ["labs_ok_observed", "labs_ok_daily", "labs_ok"])
labs_component_cols = [c for c in ["wbc_in_range", "lactate_low"] if c in df_labs.columns]

print("\nDetected columns:")
print("  phys agg:", phys_agg_col, "| phys components:", phys_component_cols)
print("  labs agg:", labs_agg_col, "| labs components:", labs_component_cols)

if phys_agg_col is None and len(phys_component_cols) == 0:
    raise ValueError("Cannot build physiology domain: no aggregated or component columns found in 07 file.")
if labs_agg_col is None and len(labs_component_cols) == 0:
    raise ValueError("Cannot build labs domain: no aggregated or component columns found in 08 file.")


Detected columns:
  phys agg: None | phys components: ['temp_ok', 'map_ok', 'sf_ok']
  labs agg: None | labs components: ['wbc_in_range', 'lactate_low']


In [20]:
# -----------------------------
# 6) Build daily observed OK flags (phys & labs)
# -----------------------------
if phys_agg_col is not None:
    df_phys_daily = df_phys[KEYS + [phys_agg_col]].copy()
    df_phys_daily = df_phys_daily.rename(columns={phys_agg_col: "phys_ok_observed"})
else:
    tmp = build_observed_ok(df_phys[KEYS + phys_component_cols], phys_component_cols)
    df_phys_daily = tmp[KEYS + ["ok_observed"]].rename(columns={"ok_observed": "phys_ok_observed"})

if labs_agg_col is not None:
    df_labs_daily = df_labs[KEYS + [labs_agg_col]].copy()
    df_labs_daily = df_labs_daily.rename(columns={labs_agg_col: "labs_ok_observed"})
else:
    tmp = build_observed_ok(df_labs[KEYS + labs_component_cols], labs_component_cols)
    df_labs_daily = tmp[KEYS + ["ok_observed"]].rename(columns={"ok_observed": "labs_ok_observed"})

df_daily = df_phys_daily.merge(df_labs_daily, on=KEYS, how="outer")

In [21]:
# -----------------------------
# 7) Define daily clinical improvement (STRICT)
# -----------------------------
# We'll treat NaN as "not observed", and only assign 1/0 when observed.
phys_obs = df_daily["phys_ok_observed"].notna()
labs_obs = df_daily["labs_ok_observed"].notna()

# observed_any: at least one domain observed
df_daily["observed_any"] = (phys_obs | labs_obs).astype(int)

# strict observed: both observed (optional; mainly relevant if REQUIRE_BOTH_DOMAINS=True)
df_daily["observed_both"] = (phys_obs & labs_obs).astype(int)

if REQUIRE_BOTH_DOMAINS:
    # To call a day "improved", require BOTH domains observed and BOTH ok==1
    df_daily["clinical_ok_observed"] = np.where(
        (phys_obs & labs_obs) & (df_daily["phys_ok_observed"] == 1) & (df_daily["labs_ok_observed"] == 1),
        1,
        np.nan
    )
else:
    # Less strict: require at least one domain observed and ok==1 in any observed domain
    df_daily["clinical_ok_observed"] = np.where(
        ((df_daily["phys_ok_observed"] == 1) | (df_daily["labs_ok_observed"] == 1)),
        1,
        np.nan
    )

# For run detection, treat NaN as 0
df_daily["clinical_ok_filled"] = df_daily["clinical_ok_observed"].fillna(0).astype(int)

print("\nDaily rates:")
print("  phys observed rate:", df_daily["phys_ok_observed"].notna().mean(), "| ok among observed:", df_daily["phys_ok_observed"].mean())
print("  labs observed rate:", df_daily["labs_ok_observed"].notna().mean(), "| ok among observed:", df_daily["labs_ok_observed"].mean())
print("  clinical observed rate:", df_daily["clinical_ok_observed"].notna().mean(), "| ok among observed:", df_daily["clinical_ok_observed"].mean())
print("  observed_both rate:", df_daily["observed_both"].mean())


Daily rates:
  phys observed rate: 0.7013024308664291 | ok among observed: 1.0
  labs observed rate: 0.6182013728173162 | ok among observed: 1.0
  clinical observed rate: 0.4411935289513545 | ok among observed: 1.0
  observed_both rate: 0.4411935289513545


In [22]:
# -----------------------------
# 8) Compute sustained response per stay
# -----------------------------
df_daily = df_daily.sort_values(["subject_id","hadm_id","icu_stay_id","day_idx"]).copy()

def per_stay_rcs(sub):
    sub = sub[sub["day_idx"] >= START_DAY_IDX].copy()
    if sub.empty:
        return pd.Series({"ever_rcs": 0, "rcs_day_idx": np.nan, "n_days_eval": 0})

    x = sub["clinical_ok_filled"].to_numpy()
    start_pos = sustained_run_first_day(x, min_consecutive=MIN_CONSECUTIVE_DAYS)

    if np.isnan(start_pos):
        return pd.Series({"ever_rcs": 0, "rcs_day_idx": np.nan, "n_days_eval": len(sub)})

    day_idx_start = int(sub.iloc[int(start_pos)]["day_idx"])
    return pd.Series({"ever_rcs": 1, "rcs_day_idx": day_idx_start, "n_days_eval": len(sub)})

df_rcs = (
    df_daily.groupby(["subject_id","hadm_id","icu_stay_id"], as_index=False)
            .apply(per_stay_rcs)
            .reset_index(drop=True)
)

print("\nRCS settings:")
print("  START_DAY_IDX:", START_DAY_IDX)
print("  MIN_CONSECUTIVE_DAYS:", MIN_CONSECUTIVE_DAYS)
print("  REQUIRE_BOTH_DOMAINS:", REQUIRE_BOTH_DOMAINS)

print("\nRCS summary:")
print("  stays:", df_rcs.shape[0])
print("  ever_rcs rate:", df_rcs["ever_rcs"].mean())
if (df_rcs["ever_rcs"] == 1).any():
    print("  median rcs_day_idx among responders:", df_rcs.loc[df_rcs["ever_rcs"]==1, "rcs_day_idx"].median())


RCS settings:
  START_DAY_IDX: 1
  MIN_CONSECUTIVE_DAYS: 3
  REQUIRE_BOTH_DOMAINS: True

RCS summary:
  stays: 21058
  ever_rcs rate: 0.3401557602811283
  median rcs_day_idx among responders: 1.0


/tmp/ipykernel_949139/16812540.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_daily.groupby(["subject_id","hadm_id","icu_stay_id"], as_index=False)


In [23]:
# -----------------------------
# 9) Attach to cohort and save
# -----------------------------
require_cols(df_cohort, ["subject_id","hadm_id","icu_stay_id"], "df_cohort")
df_cohort = to_int64(df_cohort, ["subject_id","hadm_id","icu_stay_id"])

df_out = df_cohort.merge(df_rcs, on=["subject_id","hadm_id","icu_stay_id"], how="left")
df_out["ever_rcs"] = df_out["ever_rcs"].fillna(0).astype(int)

df_out.to_parquet(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)

print(df_out[["subject_id","hadm_id","icu_stay_id","ever_rcs","rcs_day_idx"]].head(10))


Saved: 10_respuesta_clinica_sostenida.parquet
   subject_id   hadm_id  icu_stay_id  ever_rcs  rcs_day_idx
0    11239107  25883588     32367987         1          1.0
1    15811456  29271096     32876962         1          1.0
2    19326831  29957742     32456504         0          NaN
3    19326831  29957742     33950322         0          NaN
4    19326831  29957742     38112378         0          NaN
5    19727364  20803079     34384437         0          NaN
6    19727364  20803079     35521988         0          NaN
7    16708802  29996606     34034988         1          1.0
8    15952165  21530208     35549689         1          1.0
9    10913472  29471574     36636407         1          1.0


In [24]:
print("ever_rcs rate:", df_out["ever_rcs"].mean())
print(df_out.loc[df_out["ever_rcs"]==1, "rcs_day_idx"].describe())
print("Top rcs_day_idx counts:")
print(df_out.loc[df_out["ever_rcs"]==1, "rcs_day_idx"].value_counts().head(15))

ever_rcs rate: 0.4323680263221561
count    35743.000000
mean         5.086199
std          6.211895
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max         28.000000
Name: rcs_day_idx, dtype: float64
Top rcs_day_idx counts:
rcs_day_idx
1.0     16964
2.0      2915
3.0      2100
4.0      2059
5.0      1282
6.0      1221
9.0       913
7.0       784
11.0      675
8.0       671
10.0      666
12.0      606
14.0      604
15.0      488
13.0      470
Name: count, dtype: int64
